# Schema Evolution Test

Adds a new column to the source data and re-runs the Auto Loader stream to observe how it reacts to an unexpected schema change (`cloudFiles.schemaEvolutionMode = addNewColumns`).

In [0]:
%run ./lab3_00_config

In [0]:
pipeline_name = "netflix_titles_stream"

source_path = f"{storage_root}/ingestion/netflix_stream/"
checkpoint_path = f"{storage_root}/checkpoints/{pipeline_name}/"
schema_location = f"{storage_root}/schema_location/{pipeline_name}/"
bronze_table = f"{catalog}.{bronze_schema}.{pipeline_name}"

print("Source path:", source_path)
print("Checkpoint path:", checkpoint_path)
print("Schema location:", schema_location)
print("Target table:", bronze_table)

## 0. Baseline — table state before the experiment

In [0]:
before_count = spark.table(bronze_table).count()
columns_before = spark.table(bronze_table).columns

print("Row count BEFORE schema evolution test:", before_count)
print("popularity_score column exists before:", "popularity_score" in columns_before)

## 1. Add a file with a new column to the source

In [0]:
from pyspark.sql import Row

temp_path = f"{storage_root}/temp/schema_evolution/"
new_file_path = f"{source_path}netflix_new_column_test.csv"

new_data = spark.createDataFrame([
    Row(show_id="s9999", type="Movie", title="Schema Evolution Test",
        director="Test Director", cast="Test Cast", country="Testland",
        date_added="July 27, 2026", release_year="2026", rating="PG",
        duration="99 min", listed_in="Documentaries", description="A test row.",
        popularity_score="87.5")
])

# Write as a single file via a temp folder, then move it into the source folder with a clean name
dbutils.fs.rm(temp_path, recurse=True)

(
    new_data
    .coalesce(1)
    .write
    .option("header", "true")
    .csv(temp_path)
)

part_file = [f.path for f in dbutils.fs.ls(temp_path) if f.name.endswith(".csv")][0]
dbutils.fs.mv(part_file, new_file_path)
dbutils.fs.rm(temp_path, recurse=True)

print("New file with extra column written to:", new_file_path)

In [0]:
from pyspark.sql.functions import col, current_timestamp, current_date

df_bronze_stream = (
    spark.readStream
    .format("cloudFiles")
    .option("cloudFiles.format", "csv")
    .option("header", "true")
    .option("cloudFiles.schemaLocation", schema_location)
    .option("cloudFiles.schemaEvolutionMode", "addNewColumns")
    .load(source_path)
)

df_bronze_stream = (
    df_bronze_stream
    .withColumn("_source_file", col("_metadata.file_path"))
    .withColumn("_ingested_at", current_timestamp())
    .withColumn("_load_date", current_date())
)


try:
    query = (
        df_bronze_stream.writeStream
        .option("checkpointLocation", checkpoint_path)
        .option("mergeSchema", "true")
        .trigger(availableNow=True)
        .toTable(bronze_table)
    )
    query.awaitTermination()
    print("Stream completed successfully after schema evolution retry")
except Exception as e:
    print(f"Stream failed again: {type(e).__name__}: {e}")

## 2. Retry with schema evolution enabled — fix applied

In [0]:
from pyspark.sql.functions import col, current_timestamp, current_date

df_bronze_stream_retry = (
    spark.readStream
    .format("cloudFiles")
    .option("cloudFiles.format", "csv")
    .option("header", "true")
    .option("cloudFiles.schemaLocation", schema_location)
    .option("cloudFiles.schemaEvolutionMode", "addNewColumns")
    .option("rescuedDataColumn", "_rescued_data")
    .load(source_path)
)

df_bronze_stream_retry = (
    df_bronze_stream_retry
    .withColumn("_source_file", col("_metadata.file_path"))
    .withColumn("_ingested_at", current_timestamp())
    .withColumn("_load_date", current_date())
)

print("Schema after Auto Loader evolution:")
df_bronze_stream_retry.printSchema()

In [0]:
query = (
    df_bronze_stream_retry.writeStream
    .format("delta")
    .option("checkpointLocation", checkpoint_path)
    .option("mergeSchema", "true")
    .trigger(availableNow=True)
    .toTable(bronze_table)
)

query.awaitTermination()

print("✅ Stream completed successfully after schema evolution retry")

## 3. Confirm the result — after the experiment

In [0]:
after_count = spark.table(bronze_table).count()
columns_after = spark.table(bronze_table).columns

print("Row count AFTER schema evolution test:", after_count)
print("popularity_score column exists after:", "popularity_score" in columns_after)
print()
spark.table(bronze_table).printSchema()

In [0]:
# Observe the _rescued_data column for the new row specifically
spark.table(bronze_table).filter("show_id = 's9999'").select("show_id", "popularity_score", "_rescued_data").show(truncate=False)

## Summary

The `_rescued_data` value is null because `popularity_score` was successfully incorporated into the evolved schema. No data needed to be rescued for this row.